In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# === Load the trained XGBoost model ===
logged_model_uri = 'runs:/model_run_0010/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Define model input features ===
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051'
]

# === Indexing for optimization variables ===
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21,
    'REJECT_QUA': 22, 'REJECT_GCV': 23, 'MAKEUP_WATER': 24,
    'STEAM_GEN': 25, 'STEAM_ENTHALPY': 26, 'FEEDW_ENTHALPY': 27
}

# === Bounds for optimization ===
bounds = [
    (120, 150), (15, 60), (5, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (3500, 4000),
    (0.03, 0.06), (0.22, 1.17), (1.12, 1.75),
    (15.07, 31.26), (0.19, 0.37), (80.20, 296.96),
    (101.65, 185.22), (290.00, 349.98),
    (369.80, 436.67), (15.20, 49.99),
    (1000, 1300), (1300, 1500), (10, 100),
    (50, 200), (750, 1000), (100, 200)
]

# === Loss calculation function ===
def calculate_losses(x):
    Tfg, Tair, Mdfg, H2, AAS, CO, CO2, ASH, ESP_ASH, TM, AF, GCV = x[:12]
    RejectQuality = x[22]
    RejectGCV = x[23]
    MakeUpWater = x[24]
    SteamGen = x[25]
    SteamEnthalpy = x[26]
    FeedwEnthalpy = x[27]

    L1 = ((Mdfg * 0.24) * (Tfg - Tair) / GCV) * 100
    L2 = (TM * (584 + 0.45 * (Tfg - Tair))) / GCV
    L3 = 8.937 * (584 + 0.45 * (Tfg - Tair)) * H2 / GCV
    L4 = (AF * AAS * 0.45 * (Tfg - Tair) * 100) / GCV
    L5 = 0.02
    L6 = 1.13
    L7 = ((ASH * 15 / 100) * 10 * 8077) / (100 * GCV)
    L8 = ((ASH * 80 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L9 = ((ASH * 20 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L10 = 0.65
    L11 = ((0.8 * MakeUpWater * 300 + 0.2 * MakeUpWater * 150) * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) / 0.85)
    L12 = (RejectQuality * RejectGCV * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) * 1000 / 0.85)

    UL = L6 + L7
    SL = L8 + L9
    total = L1 + L2 + L3 + L4 + L5 + UL + SL + L10 + L11 + L12

    return total, {
        'Total Loss': total, 'L1': L1, 'L2': L2, 'L3': L3, 'L4': L4, 'L5': L5,
        'L6': L6, 'L7': L7, 'L8': L8, 'L9': L9, 'L10': L10, 'L11': L11, 'L12': L12
    }

# === Monte Carlo Loss ===
def calculate_loss_ranges(n_samples=5000):
    'Calculate min/max for each loss component through Monte Carlo sampling'
    loss_ranges = {f'L{i}': [np.inf, -np.inf] for i in range(1,13)}
    loss_ranges['Total Loss'] = [np.inf, -np.inf]

    for _ in range(n_samples):
        sample = [np.random.uniform(low, high) for (low, high) in bounds]
        try:
            total_loss, loss_dict = calculate_losses(sample)
            for name, value in loss_dict.items():
                loss_ranges[name][0] = min(loss_ranges[name][0], value)
                loss_ranges[name][1] = max(loss_ranges[name][1], value)
        except Exception:
            continue
    return loss_ranges

# === Print Loss Ranges as (min, max) ===
def print_loss_ranges(loss_ranges):
    print('\n🔥 Loss Value Ranges (Min, Max):')
    for loss in ['Total Loss'] + [f'L{i}' for i in range(1,13)]:
        min_val, max_val = loss_ranges.get(loss, [0,0])
        print(f"{loss}: ({min_val:.3f}, {max_val:.3f})")

# === Main Execution ===
if __name__ == '__main__':
    # 1. Run optimization (your original code)
    n_runs = 20
    optimized_solutions = []
    for i in range(n_runs):
        x0 = np.array([np.random.uniform(low, high) for (low, high) in bounds])
        res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=[
            {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
            {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
            {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
        ])
        if res.success:
            optimized_solutions.append(res.x)

    if not optimized_solutions:
        raise RuntimeError('All optimization attempts failed.')

    optimized_array = np.array(optimized_solutions)
    print('\n📈 Optimal Input Variable Ranges (Based on Multiple Runs):')
    for i, (low, high) in enumerate(bounds):
        opt_min = optimized_array[:, i].min()
        opt_max = optimized_array[:, i].max()
        print(f"Var {i} - Optimum Min: {opt_min:.3f}, Max: {opt_max:.3f}, Original Bounds: ({low}, {high})")

    best_idx = np.argmax([-objective(x) for x in optimized_solutions])
    x_opt = optimized_solutions[best_idx]
    efficiency = -objective(x_opt)
    _, loss_dict = calculate_losses(x_opt)

    print(f"\n✅ Best Boiler Efficiency Achieved: {efficiency:.2f}%")
    print('\n📊 Optimal Input Variables:')
    for name, idx in var_index.items():
        print(f"{name}: {x_opt[idx]:.3f}")
    for name, idx in additional_index.items():
        print(f"{name}: {x_opt[idx]:.3f}")

    print('\n🔥 Loss Breakdown (Optimized Point):')
    for name, value in loss_dict.items():
        print(f"{name}: {value:.3f}")

    # === Print Loss Ranges Across Envelope as (min, max) ===
    loss_ranges = calculate_loss_ranges(2000)
    print_loss_ranges(loss_ranges)


In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# === Load the trained XGBoost model ===
logged_model_uri = 'runs:/model_run_0010/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Define model input features ===
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051'
]

# === Indexing for optimization variables ===
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21,
    'REJECT_QUA': 22, 'REJECT_GCV': 23, 'MAKEUP_WATER': 24,
    'STEAM_GEN': 25, 'STEAM_ENTHALPY': 26, 'FEEDW_ENTHALPY': 27
}

# === Bounds for optimization ===
bounds = [
    (120, 150), (15, 60), (5, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (3500, 4000),
    (0.03, 0.06), (0.22, 1.17), (1.12, 1.75),
    (15.07, 31.26), (0.19, 0.37), (80.20, 296.96),
    (101.65, 185.22), (290.00, 349.98),
    (369.80, 436.67), (15.20, 49.99),
    (1000, 1300), (1300, 1500), (10, 100),
    (50, 200), (750, 1000), (100, 200)
]

# === Loss calculation function ===
def calculate_losses(x):
    Tfg, Tair, Mdfg, H2, AAS, CO, CO2, ASH, ESP_ASH, TM, AF, GCV = x[:12]
    RejectQuality = x[22]
    RejectGCV = x[23]
    MakeUpWater = x[24]
    SteamGen = x[25]
    SteamEnthalpy = x[26]
    FeedwEnthalpy = x[27]

    L1 = ((Mdfg * 0.24) * (Tfg - Tair) / GCV) * 100
    L2 = (TM * (584 + 0.45 * (Tfg - Tair))) / GCV
    L3 = 8.937 * (584 + 0.45 * (Tfg - Tair)) * H2 / GCV
    L4 = (AF * AAS * 0.45 * (Tfg - Tair) * 100) / GCV
    L5 = 0.02  # Updated to match your constraint
    L6 = 1.13
    L7 = ((ASH * 15 / 100) * 10 * 8077) / (100 * GCV)
    L8 = ((ASH * 80 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L9 = ((ASH * 20 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L10 = 0.40  # Updated to match your constraint
    L11 = ((0.8 * MakeUpWater * 300 + 0.2 * MakeUpWater * 150) * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) / 0.85)
    L12 = (RejectQuality * RejectGCV * 100) / ((SteamGen * (SteamEnthalphy - FeedwEnthalpy)) * 1000 / 0.85)

    UL = L6 + L7
    SL = L8 + L9
    total = L1 + L2 + L3 + L4 + L5 + UL + SL + L10 + L11 + L12

    return total, {
        'Total Loss': total, 'L1': L1, 'L2': L2, 'L3': L3, 'L4': L4, 'L5': L5,
        'L6': L6, 'L7': L7, 'L8': L8, 'L9': L9, 'L10': L10, 'L11': L11, 'L12': L12,
        'UL': UL, 'SL': SL
    }

def loss_constraints(x):
    total_loss, loss_dict = calculate_losses(x)
    constraints = []
    
    
    constraints.append(4.81 - loss_dict['L1'])    
    constraints.append(2.15 - loss_dict['L2'])    
    constraints.append(4.17 - loss_dict['L3'])    
    constraints.append(0.14 - loss_dict['L4'])    
    constraints.append(0.02 - loss_dict['L5'])    
    
    UL = loss_dict['L6'] + loss_dict['L7']
    constraints.append(1.47 - UL)                 
    
    SL = loss_dict['L8'] + loss_dict['L9']
    constraints.append(0.29 - SL)                 
    
    constraints.append(0.40 - loss_dict['L10'])   
    constraints.append(0.07 - loss_dict['L11'])
    constraints.append(0.13 - loss_dict['L12'])
    return constraints

# === Objective Function ===
def objective(x):
    'Minimize total loss (maximize efficiency)'
    total_loss, _ = calculate_losses(x)
    return total_loss

# === Monte Carlo Loss Ranges ===
def calculate_loss_ranges(n_samples=5000):
    'Calculate min/max for each loss component through Monte Carlo sampling'
    loss_ranges = {f'L{i}': [np.inf, -np.inf] for i in range(1,13)}
    loss_ranges.update({'Total Loss': [np.inf, -np.inf], 'UL': [np.inf, -np.inf], 'SL': [np.inf, -np.inf]})

    for _ in range(n_samples):
        sample = [np.random.uniform(low, high) for (low, high) in bounds]
        try:
            total_loss, loss_dict = calculate_losses(sample)
            for name, value in loss_dict.items():
                if name in loss_ranges:
                    loss_ranges[name][0] = min(loss_ranges[name][0], value)
                    loss_ranges[name][1] = max(loss_ranges[name][1], value)
        except Exception:
            continue
    return loss_ranges

# === Print Loss Ranges as (min, max) ===
def print_loss_ranges(loss_ranges):
    print('\n🔥 Loss Value Ranges (Min, Max):')
    for loss in ['Total Loss'] + [f'L{i}' for i in range(1,13)] + ['UL', 'SL']:
        if loss in loss_ranges:
            min_val, max_val = loss_ranges[loss]
            print(f"{loss}: ({min_val:.3f}, {max_val:.3f})")

# === Print Constraint Validation ===
def print_constraint_validation(loss_ranges):
    'Check which constraints are satisfied across the range'
    design_limits = {
        'L1': 4.81, 'L2': 2.15, 'L3': 4.17, 'L4': 0.14, 'L5': 0.02,
        'UL': 1.47, 'SL': 0.29, 'L10': 0.40
    }
    
    print('\n🎯 Design Constraint Validation:')
    for loss_name, limit in design_limits.items():
        if loss_name in loss_ranges:
            min_val, max_val = loss_ranges[loss_name]
            if max_val <= limit:
                status = '✅ ALWAYS SATISFIED'
            elif min_val > limit:
                status = '❌ ALWAYS VIOLATED'
            else:
                status = '⚠️ SOMETIMES VIOLATED'
            print(f"{loss_name}: Range({min_val:.3f}, {max_val:.3f}) vs Limit({limit}) - {status}")

# === Main Execution ===
if __name__ == '__main__':
    print('🚀 Starting Constrained Boiler Optimization with Design Loss Limits...')
    
    # 1. Run constrained optimization
    n_runs = 20
    optimized_solutions = []
    constraint_violations = []
    
    for i in range(n_runs):
        x0 = np.array([np.random.uniform(low, high) for (low, high) in bounds])
        
        # Original constraints + Design Loss Constraints
        all_constraints = [
            {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
            {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
            {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
        ]
        
        # Add individual loss constraints
        for j in range(8):  # 8 loss constraints
            all_constraints.append({
                'type': 'ineq', 
                'fun': lambda x, idx=j: loss_constraints(x)[idx]
            })
        
        res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=all_constraints)
        
        if res.success:
            optimized_solutions.append(res.x)
        else:
            # Check constraint violations for failed optimizations
            constraints_vals = loss_constraints(x0)
            constraint_violations.append(constraints_vals)

    print(f"\n📊 Optimization Results:")
    print(f"Successful optimizations: {len(optimized_solutions)}/{n_runs}")
    print(f"Failed optimizations: {len(constraint_violations)}")

    if optimized_solutions:
        optimized_array = np.array(optimized_solutions)
        print('\n📈 Optimal Input Variable Ranges (Constraint-Satisfying Solutions):')
        for i, (low, high) in enumerate(bounds):
            opt_min = optimized_array[:, i].min()
            opt_max = optimized_array[:, i].max()
            print(f"Var {i:2d} - Feasible Range: ({opt_min:7.3f}, {opt_max:7.3f}) | Original: ({low:7.2f}, {high:7.2f})")

        # Best solution
        best_idx = np.argmin([objective(x) for x in optimized_solutions])
        x_opt = optimized_solutions[best_idx]
        efficiency = 100 - objective(x_opt)
        _, loss_dict = calculate_losses(x_opt)

        print(f"\n✅ Best Constrained Boiler Efficiency: {efficiency:.2f}%")
        print(f"Total Loss: {objective(x_opt):.3f}%")

        print('\n🔥 Optimal Loss Breakdown:')
        design_limits = {'L1': 4.81, 'L2': 2.15, 'L3': 4.17, 'L4': 0.14, 'L5': 0.02, 'UL': 1.47, 'SL': 0.29, 'L10': 0.40}
        for name, value in loss_dict.items():
            if name in design_limits:
                limit = design_limits[name]
                status = '✅' if value <= limit else '❌'
                print(f"{status} {name}: {value:.3f}% (Limit: {limit}%)")
            elif name not in ['Total Loss']:
                print(f"ℹ️ {name}: {value:.3f}%")

    # 2. Calculate loss ranges across entire input space
    print('\n🔄 Calculating Loss Ranges Across Input Space...')
    loss_ranges = calculate_loss_ranges(10000)
    print_loss_ranges(loss_ranges)
    
    # 3. Validate constraints across ranges
    print_constraint_validation(loss_ranges)


In [ ]:
Loss Component Ranges:
Total Loss: (17.13, 18.93)%
L1 (Dry Flue Gas): (4.84, 5.35)% ❌ > 4.81% limit
L2 (H₂O in fuel): (1.78, 1.97)% ✅ < 2.15% limit  
L3 (H₂ in fuel): (2.89, 3.19)% ✅ < 4.17% limit
L4 (H₂O in Air): (0.49, 0.54)% ❌ > 0.14% limit
L5 (Partial Combustion): (0.019, 0.021)% ✅ < 0.02% limit
UL (Total Unburnt): (1.83, 2.02)% ❌ > 1.47% limit
SL (Total Sensible): (0.13, 0.15)% ✅ < 0.29% limit
L10 (Radiation): (0.62, 0.68)% ❌ > 0.40% limit